<a href="https://colab.research.google.com/github/SiramGanesh/ML-DEEP-LEARNING-QUANTUM-ML/blob/main/quantum_cic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import os
from google.colab import drive
import time

# Mount Google Drive
drive.mount('/content/drive')

# Define the path to your folder
folder_path = '/content/drive/My Drive/cic_iot_2023_dataset/cic_iot_2023'

# Check if the folder exists
if not os.path.exists(folder_path):
    print(f"Error: Folder '{folder_path}' not found. Please check the folder name and path in your Google Drive.")
else:
    print(f"Contents of '{folder_path}':")
    files = os.listdir(folder_path)
    if files:
        for f in files:
            print(f)

        # Assuming the data is in a CSV file, let's try to find and load the first one
        csv_files = [f for f in files if f.endswith('.csv')]
        if csv_files:
            file_to_load = os.path.join(folder_path, csv_files[0])
            print(f"\nAttempting to load '{file_to_load}' into DataFrame 'df'...")
            try:
                df = pd.read_csv(file_to_load)
                print("DataFrame 'df' loaded successfully. First 5 rows:")
                print(df.head())
            except Exception as e:
                print(f"Error loading CSV file: {e}")
        else:
            print("No CSV files found in the folder. Please specify the correct file type or name.")
    else:
        print("The folder is empty.")


ValueError: mount failed

In [ ]:
dfs = []
SAMPLES_PER_FILE = 3000  # Adjust as needed

# Ensure 'files' is sorted for consistent processing if order matters
# files.sort() # Uncomment if you need a specific order

print(f"Processing {len(csv_files)} CSV files...")

for i, file in enumerate(csv_files):
    full_file_path = os.path.join(folder_path, file)
    try:
        temp = pd.read_csv(full_file_path)

        sample_size = min(SAMPLES_PER_FILE, len(temp))

        temp_sampled = temp.sample(
            n=sample_size,
            random_state=42
        )

        dfs.append(temp_sampled)

        if (i + 1) % 20 == 0:
            print(f"Processed {i+1}/{len(csv_files)} files")
    except Exception as e:
        print(f"Error processing file {full_file_path}: {e}")

df = pd.concat(dfs, ignore_index=True)

print("\nFinal DataFrame 'df' created by concatenating samples.")
print("Final Shape:", df.shape)

print("\nNumber of Classes (unique labels):")
print(df['label'].nunique())

print("\nTop 10 Labels (value counts):")
print(df['label'].value_counts().head(10))

In [ ]:
print("Total Classes:", df['label'].nunique())

print("\nClasses:")
print(df['label'].value_counts())


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["multi_label"] = le.fit_transform(df["label"])

print("Number of classes:", df["multi_label"].nunique())

print("\nEncoded Classes:")

for i, cls in enumerate(le.classes_):
    print(i, "->", cls)

In [ ]:
import numpy as np

print("Dataset Shape:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum().sum())

print("\nInfinite Values:")
print(np.isinf(df.select_dtypes(include=[np.number])).sum().sum())

print("\nData Types:")
print(df.dtypes)

In [ ]:
feature_cols = [col for col in df.columns
                if col not in ["label",
                               "multi_label"]]

print("Number of Features:", len(feature_cols))

print(feature_cols)

In [ ]:
X = df[feature_cols].copy()
y_multi = df["multi_label"].copy()

print("X Shape:", X.shape)
print("Multiclass Labels:", y_multi.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_multi,
    test_size=0.30,
    random_state=42,
    stratify=y_multi
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

print("\nClasses in Train:", y_train.nunique())
print("Classes in Test:", y_test.nunique())

In [ ]:
!pip install qiskit qiskit-machine-learning

In [ ]:
!pip install qiskit-algorithms

In [ ]:
!pip install tensorflow-quantum

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

print(X_train.shape)
print(X_test.shape)

In [ ]:
# Classical ML
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

# Quantum ML
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_algorithms.optimizers import COBYLA
from qiskit_machine_learning.algorithms import VQC

In [ ]:
# Train classical SVM model
from sklearn.svm import SVC

svm = SVC(C=0.1, gamma='scale', kernel='rbf')
svm.fit(X_train, y_train)

# Predict on test data
y_pred = svm.predict(X_test)

# Print results
print("===== Classical SVM Results =====")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
import tensorflow as tf
import tensorflow_quantum as tfq
import cirq
import sympy
import numpy as np
from sklearn.decomposition import PCA

In [ ]:
# 1. Preprocessing: Reduce 46 features to 6 for Quantum compatibility
# You must do this BEFORE converting to circuits
pca = PCA(n_components=6)
X_train_reduced = pca.fit_transform(X_train)

In [ ]:
# 2. Conversion Function
def convert_data_to_circuits(X_data):
    """Maps reduced features to rotation gates on qubits."""
    num_qubits = X_data.shape[1]
    qubits = cirq.GridQubit.rect(1, num_qubits)
    circuits = []
    for row in X_data:
        circuit = cirq.Circuit()
        for i, val in enumerate(row):
            # val is already scaled [0, 1] by MinMaxScaler
            circuit.append(cirq.rx(val * np.pi)(qubits[i]))
        circuits.append(circuit)
    return circuits

In [ ]:
# 3. Create a manageable subset for the quantum experiment
# Training on 700k rows with quantum circuits is not feasible on current hardware
subset_size = 100000
tfq_x_train = tfq.convert_to_tensor(convert_data_to_circuits(X_train_reduced[:subset_size]))

In [ ]:
import sympy
import tensorflow as tf
import tensorflow_quantum as tfq
import cirq
import numpy as np

# 4. Build the Hybrid Quantum-Classical Model using a Custom Layer
# This works around the TypeError: Layer.add_weight() got multiple values for argument 'shape'

num_qubits = 6
qubits = cirq.GridQubit.rect(1, num_qubits)
params = [sympy.Symbol(f'theta{i}') for i in range(num_qubits)]
symbolic_circuit = cirq.Circuit([cirq.rx(p)(qubits[i]) for i, p in enumerate(params)])
measurement_operator = cirq.Z(qubits[0])

class CustomQuantumLayer(tf.keras.layers.Layer):
    def __init__(self, output_dim, **kwargs):
        super(CustomQuantumLayer, self).__init__(**kwargs)
        self.output_dim = output_dim
        self.expectation_layer = tfq.layers.Expectation()

    def build(self, input_shape):
        # Manually manage the weights to avoid the PQC internal bug
        self.kernel = self.add_weight(
            name='kernel',
            shape=(len(params),),
            initializer=tf.keras.initializers.RandomUniform(minval=0, maxval=2*np.pi),
            trainable=True
        )
        self.dense_layer = tf.keras.layers.Dense(self.output_dim, activation='softmax')

    def call(self, inputs):
        # inputs are the circuit tensors
        # Map our trainable kernel weights to the symbols in the circuit
        param_values = tf.tile(tf.expand_dims(self.kernel, 0), [tf.shape(inputs)[0], 1])

        # Resolve and compute expectation
        quantum_output = self.expectation_layer(
            inputs,
            symbol_names=[str(p) for p in params],
            symbol_values=param_values,
            operators=measurement_operator
        )

        return self.dense_layer(quantum_output)

# Rebuild the model
quantum_layer = CustomQuantumLayer(output_dim=34)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(), dtype=tf.string),
    quantum_layer
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.02),
              loss=tf.keras.losses.SparseCategoricalCrossentropy())

In [ ]:
# 5. Train
model.fit(tfq_x_train, y_train[:subset_size], epochs=5, batch_size=32)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# 1. Prepare the test subset
# We use the same PCA already fitted on X_train
X_test_reduced = pca.transform(X_test[:subset_size])
tfq_x_test = tfq.convert_to_tensor(convert_data_to_circuits(X_test_reduced))

# 2. Get predictions from the hybrid model
# The output is a probability distribution (due to softmax)
y_pred_prob = model.predict(tfq_x_test)
y_pred_quantum = np.argmax(y_pred_prob, axis=1)

# 3. Calculate Accuracy
quantum_accuracy = accuracy_score(y_test[:subset_size], y_pred_quantum)

print(f"===== Hybrid Quantum Model Results (Subset Size: {subset_size}) =====")
print(f"Accuracy: {quantum_accuracy:.4f}")

# Detailed report for the subset
print("\nClassification Report:")
print(classification_report(y_test[:subset_size], y_pred_quantum, zero_division=0))

### Implementing QCNN (Quantum Convolutional Neural Network)
Based on the TensorFlow Quantum QCNN tutorial, we define local filters (convolutions) and dimensionality reduction (pooling) using quantum gates.

In [ ]:
def one_qubit_unitary(bit, symbols):
    return cirq.Circuit(
        cirq.X(bit)**symbols[0],
        cirq.Y(bit)**symbols[1],
        cirq.Z(bit)**symbols[2])

def two_qubit_unitary(bits, symbols):
    circuit = cirq.Circuit()
    circuit += one_qubit_unitary(bits[0], symbols[0:3])
    circuit += one_qubit_unitary(bits[1], symbols[3:6])
    circuit += [cirq.ZZ(*bits)**symbols[6]]
    circuit += [cirq.YY(*bits)**symbols[7]]
    circuit += [cirq.XX(*bits)**symbols[8]]
    circuit += one_qubit_unitary(bits[0], symbols[9:12])
    circuit += one_qubit_unitary(bits[1], symbols[12:15])
    return circuit

def quantum_conv_circuit(bits, symbols):
    circuit = cirq.Circuit()
    for first, second in zip(bits[0::2], bits[1::2]):
        circuit += two_qubit_unitary([first, second], symbols)
    for first, second in zip(bits[1::2], bits[2::2] + [bits[0]]):
        circuit += two_qubit_unitary([first, second], symbols)
    return circuit

def quantum_pool_circuit(source_bits, sink_bits, symbols):
    circuit = cirq.Circuit()
    for source, sink in zip(source_bits, sink_bits):
        circuit += two_qubit_unitary([source, sink], symbols)
    return circuit

In [ ]:
# Define QCNN structure for 6 qubits
qcnn_qubits = cirq.GridQubit.rect(1, 6)
qcnn_model_circuit = cirq.Circuit()

# Symbols for different layers
conv_symbols = sympy.symbols('conv0:15')
pool_symbols = sympy.symbols('pool0:15')

# 1. Convolution Layer
qcnn_model_circuit += quantum_conv_circuit(qcnn_qubits, conv_symbols)

# 2. Pooling Layer (Reduce 6 to 3 qubits)
qcnn_model_circuit += quantum_pool_circuit(qcnn_qubits[:3], qcnn_qubits[3:], pool_symbols)

# Output operator (Readout from one of the remaining qubits)
readout_operators = [cirq.Z(qcnn_qubits[i]) for i in [3, 4, 5]]

# Using the workaround logic from before to avoid the add_weight shape bug
class QCNNLayer(tf.keras.layers.Layer):
    def __init__(self, output_dim, **kwargs):
        super(QCNNLayer, self).__init__(**kwargs)
        self.output_dim = output_dim
        self.expectation_layer = tfq.layers.Expectation()

    def build(self, input_shape):
        self.conv_weights = self.add_weight(name='conv_weights', shape=(15,), initializer='random_normal', trainable=True)
        self.pool_weights = self.add_weight(name='pool_weights', shape=(15,), initializer='random_normal', trainable=True)
        self.dense = tf.keras.layers.Dense(self.output_dim, activation='softmax')

    def call(self, inputs):
        batch_dim = tf.shape(inputs)[0]
        full_params = tf.concat([self.conv_weights, self.pool_weights], axis=0)
        param_values = tf.tile(tf.expand_dims(full_params, 0), [batch_dim, 1])

        # symbols must match the order in concat
        all_symbols = [str(s) for s in conv_symbols] + [str(s) for s in pool_symbols]

        out = self.expectation_layer(
            inputs,
            symbol_names=all_symbols,
            symbol_values=param_values,
            operators=readout_operators
        )
        return self.dense(out)

qcnn_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(), dtype=tf.string),
    QCNNLayer(output_dim=34)
])

qcnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train on the same subset
qcnn_model.fit(tfq_x_train, y_train[:subset_size], epochs=5, batch_size=64)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# 1. Prepare the test data for QCNN
eval_subset = 10000
X_test_reduced_qcnn = pca.transform(X_test[:eval_subset])
tfq_x_test_qcnn = tfq.convert_to_tensor(convert_data_to_circuits(X_test_reduced_qcnn))

# 2. Get predictions from the QCNN model
y_pred_qcnn_prob = qcnn_model.predict(tfq_x_test_qcnn)
y_pred_qcnn = np.argmax(y_pred_qcnn_prob, axis=1)

# 3. Calculate and display Accuracy
qcnn_accuracy = accuracy_score(y_test[:eval_subset], y_pred_qcnn)

print(f"===== QCNN Model Results (Test Subset: {eval_subset}) =====")
print(f"Accuracy: {qcnn_accuracy:.4f}")

# Detailed Classification Report
# We filter the target names to only those present in the subset to avoid the ValueError
unique_labels_in_subset = np.unique(np.concatenate([y_test[:eval_subset], y_pred_qcnn]))

print("\nClassification Report:")
print(classification_report(
    y_test[:eval_subset],
    y_pred_qcnn,
    labels=unique_labels_in_subset,
    target_names=le.classes_[unique_labels_in_subset],
    zero_division=0
))

### Deeper QCNN Architecture
We are now implementing a multi-stage QCNN with two layers of convolutions and pooling to improve feature extraction.

In [ ]:
# Define new symbols for the deeper layers
conv1_symbols = sympy.symbols('conv1_0:15')
pool1_symbols = sympy.symbols('pool1_0:15')
conv2_symbols = sympy.symbols('conv2_0:15')
pool2_symbols = sympy.symbols('pool2_0:15')

class DeeperQCNNLayer(tf.keras.layers.Layer):
    def __init__(self, output_dim, **kwargs):
        super(DeeperQCNNLayer, self).__init__(**kwargs)
        self.output_dim = output_dim
        self.expectation_layer = tfq.layers.Expectation()

    def build(self, input_shape):
        # Weights for Stage 1 (6 to 3 qubits)
        self.c1 = self.add_weight(name='c1', shape=(15,), initializer='random_normal')
        self.p1 = self.add_weight(name='p1', shape=(15,), initializer='random_normal')
        # Weights for Stage 2 (3 to 1 qubit)
        self.c2 = self.add_weight(name='c2', shape=(15,), initializer='random_normal')
        self.p2 = self.add_weight(name='p2', shape=(15,), initializer='random_normal')

        # Deeper classical head
        self.hidden_dense = tf.keras.layers.Dense(64, activation='relu')
        self.final_dense = tf.keras.layers.Dense(self.output_dim, activation='softmax')

    def call(self, inputs):
        batch_dim = tf.shape(inputs)[0]
        full_params = tf.concat([self.c1, self.p1, self.c2, self.p2], axis=0)
        param_values = tf.tile(tf.expand_dims(full_params, 0), [batch_dim, 1])

        all_symbols = [str(s) for s in list(conv1_symbols) + list(pool1_symbols) + list(conv2_symbols) + list(pool2_symbols)]

        # Output from the last qubit after reduction
        out = self.expectation_layer(
            inputs,
            symbol_names=all_symbols,
            symbol_values=param_values,
            operators=cirq.Z(qcnn_qubits[5])
        )

        x = self.hidden_dense(out)
        return self.final_dense(x)

deeper_qcnn = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(), dtype=tf.string),
    DeeperQCNNLayer(output_dim=34)
])

deeper_qcnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Training on the subset
deeper_qcnn.fit(tfq_x_train, y_train[:subset_size], epochs=10, batch_size=128)

In [ ]:
# Evaluate the deeper model
y_pred_deep = np.argmax(deeper_qcnn.predict(tfq_x_test_qcnn), axis=1)
deep_acc = accuracy_score(y_test[:eval_subset], y_pred_deep)
print(f"Deeper QCNN Accuracy: {deep_acc:.4f}")

### Optimization: Multi-headed Readout QCNN
To fix the performance drop, we will now measure multiple qubits from the final stage to provide a richer feature set to the classical dense layer.

In [ ]:
class MultiHeadQCNNLayer(tf.keras.layers.Layer):
    def __init__(self, output_dim, **kwargs):
        super(MultiHeadQCNNLayer, self).__init__(**kwargs)
        self.output_dim = output_dim
        self.expectation_layer = tfq.layers.Expectation()

    def build(self, input_shape):
        self.c1 = self.add_weight(name='c1', shape=(15,), initializer='random_normal')
        self.p1 = self.add_weight(name='p1', shape=(15,), initializer='random_normal')
        self.c2 = self.add_weight(name='c2', shape=(15,), initializer='random_normal')
        self.p2 = self.add_weight(name='p2', shape=(15,), initializer='random_normal')

        self.hidden_dense = tf.keras.layers.Dense(128, activation='relu')
        self.final_dense = tf.keras.layers.Dense(self.output_dim, activation='softmax')

    def call(self, inputs):
        batch_dim = tf.shape(inputs)[0]
        full_params = tf.concat([self.c1, self.p1, self.c2, self.p2], axis=0)
        param_values = tf.tile(tf.expand_dims(full_params, 0), [batch_dim, 1])

        all_symbols = [str(s) for s in list(conv1_symbols) + list(pool1_symbols) + list(conv2_symbols) + list(pool2_symbols)]

        # Measure 3 qubits instead of 1 to get more feature variance
        multi_readout = [cirq.Z(qcnn_qubits[i]) for i in [3, 4, 5]]

        out = self.expectation_layer(
            inputs,
            symbol_names=all_symbols,
            symbol_values=param_values,
            operators=multi_readout
        )

        x = self.hidden_dense(out)
        return self.final_dense(x)

optimized_qcnn = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(), dtype=tf.string),
    MultiHeadQCNNLayer(output_dim=34)
])

optimized_qcnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.002),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Increased epochs for the optimized architecture
optimized_qcnn.fit(tfq_x_train, y_train[:subset_size], epochs=15, batch_size=128)

In [ ]:
y_pred_opt = np.argmax(optimized_qcnn.predict(tfq_x_test_qcnn), axis=1)
opt_acc = accuracy_score(y_test[:eval_subset], y_pred_opt)
print(f"Optimized Multi-Head QCNN Accuracy: {opt_acc:.4f}")

### Label Reduction: Mapping 34 Classes to 4 Categories
We will group the labels into: `DDoS`, `DoS`, `Mirai`, and `Benign/Other` to simplify the learning task.

### Classical CNN Model for 4-Class Classification

In [ ]:
import time

# Define the classical CNN model for 4 classes
cnn_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_reduced.shape[1],)), # Input is 6 features
    tf.keras.layers.Reshape((X_train_reduced.shape[1], 1)), # Reshape for Conv1D
    tf.keras.layers.Conv1D(filters=32, kernel_size=3, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling1D(pool_size=2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(4, activation='softmax') # 4 output classes
])

cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.002),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Classical CNN Model Summary:")
cnn_model.summary()

# Get the subset sizes used for QCNN for fair comparison
# These variables were defined earlier in the notebook
# subset_size from cell jfyE0eQAdvBh
# eval_subset from cell 922dfafe

# Using the already defined subset_size (10000) and eval_subset (10000)

# Train the classical CNN and measure time
print("\nTraining Classical CNN...")
training_start_time_cnn = time.time()
cnn_model.fit(X_train_reduced[:subset_size], y_train_4, epochs=10, batch_size=128, verbose=0)
training_end_time_cnn = time.time()
classical_cnn_training_time = training_end_time_cnn - training_start_time_cnn
print(f"Classical CNN Training Time: {classical_cnn_training_time:.2f} seconds")

# Predict with classical CNN and measure time
print("\nPredicting with Classical CNN...")
prediction_start_time_cnn = time.time()
y_pred_cnn_4 = np.argmax(cnn_model.predict(X_test_reduced[:eval_subset]), axis=1)
prediction_end_time_cnn = time.time()
classical_cnn_prediction_time = prediction_end_time_cnn - prediction_start_time_cnn
print(f"Classical CNN Prediction Time: {classical_cnn_prediction_time:.2f} seconds")

# Evaluate classical CNN
acc_cnn_4 = accuracy_score(y_test_4, y_pred_cnn_4)
print(f"\nClassical CNN Accuracy (4 Classes): {acc_cnn_4:.4f}")

print("\nClassical CNN Classification Report (4 Classes):")
print(classification_report(y_test_4, y_pred_cnn_4, target_names=['DDoS', 'DoS', 'Mirai', 'Benign/Other']))


In [ ]:
def map_to_4_classes(label_str):
    label_str = str(label_str).lower()
    if 'ddos' in label_str:
        return 0 # DDoS
    elif 'dos' in label_str:
        return 1 # DoS
    elif 'mirai' in label_str:
        return 2 # Mirai
    elif 'benign' in label_str:
        return 3 # Benign
    else:
        return 3 # Grouping rare attacks/others into Benign for simplicity

# Apply mapping to the original labels
y_train_4 = np.array([map_to_4_classes(le.inverse_transform([val])[0]) for val in y_train[:subset_size]])
y_test_4 = np.array([map_to_4_classes(le.inverse_transform([val])[0]) for val in y_test[:eval_subset]])

print("New Class Distribution (Train):", np.unique(y_train_4, return_counts=True))
print("Mapping: 0:DDoS, 1:DoS, 2:Mirai, 3:Benign/Other")

In [ ]:
import time

# Re-initialize the model for 4 output classes
qcnn_4_class = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(), dtype=tf.string),
    MultiHeadQCNNLayer(output_dim=4)
])

qcnn_4_class.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.002),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train the 4-class model and measure time
print("\nTraining 4-Class QCNN...")
training_start_time_qcnn = time.time()
qcnn_4_class.fit(tfq_x_train, y_train_4, epochs=10, batch_size=128, verbose=0)
training_end_time_qcnn = time.time()
qcnn_training_time = training_end_time_qcnn - training_start_time_qcnn
print(f"4-Class QCNN Training Time: {qcnn_training_time:.2f} seconds")

In [ ]:
# Evaluate the 4-class model
print("\nPredicting with 4-Class QCNN...")
prediction_start_time_qcnn = time.time()
y_pred_4 = np.argmax(qcnn_4_class.predict(tfq_x_test_qcnn, verbose=0), axis=1)
prediction_end_time_qcnn = time.time()
qcnn_prediction_time = prediction_end_time_qcnn - prediction_start_time_qcnn
print(f"4-Class QCNN Prediction Time: {qcnn_prediction_time:.2f} seconds")

acc_4 = accuracy_score(y_test_4, y_pred_4)
print(f"4-Class QCNN Accuracy: {acc_4:.4f}")

print("\nClassification Report (4 Classes):")
print(classification_report(y_test_4, y_pred_4, target_names=['DDoS', 'DoS', 'Mirai', 'Benign/Other']))

In [ ]:
import pandas as pd

data = {
    'Model': ['Classical CNN', 'QCNN (4-Class)'],
    'Training Time (seconds)': [classical_cnn_training_time, qcnn_training_time]
}

training_times_df = pd.DataFrame(data)
print(training_times_df.to_markdown(index=False))